### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sat11_hand_algo_runtime",
    dataset_year="2011",
    domain_str="technology & internet",
    # Data Source
    dataset_source="ASlib",
    original_dataset_source_download_link="https://github.com/coseal/aslib_data/tree/master/SAT11-HAND",
    download_description="""
wget https://raw.githubusercontent.com/coseal/aslib_data/refs/heads/master/SAT11-HAND-ALGO/feature_values.arff \
&& wget https://raw.githubusercontent.com/coseal/aslib_data/refs/heads/master/SAT11-HAND-ALGO/algorithm_feature_values.arff \
&& wget https://raw.githubusercontent.com/coseal/aslib_data/refs/heads/master/SAT11-HAND-ALGO/algorithm_runs.arff \
&& mkdir -p local-data-warehouse/sat11_hand_algo_runtime \
&& mv feature_values.arff algorithm_feature_values.arff algorithm_runs.arff local-data-warehouse/sat11_hand_algo_runtime/
""",
    # References
    academic_reference_bibtex="""@inproceedings{xu-sat12a,
  author    = {L. Xu and F. Hutter and H. Hoos and K. Leyton-Brown},
  title     = {Evaluating Component Solver Contributions to Portfolio-Based Algorithm Selectors},
  pages     = {228-241},
  crossref  = {sat12}
}
@Proceedings{sat12,
  editor =         {A. Cimatti and R. Sebastiani},
  title =         {Proceedings of the Fifteenth International Conference on Theory and Applications of Satisfiability Testing (SAT'12)},
  booktitle = {Proceedings of the Fifteenth International Conference on Theory and Applications of Satisfiability Testing (SAT'12)},
  publisher =         springer,
  series =         lncs,
  volume =         7317,
  year =         2012
}
@article{bischl_aslib_2016,
	title = {{ASlib}: {A} {Benchmark} {Library} for {Algorithm} {Selection}},
	number = {237},
	journal = {Artificial Intelligence Journal (AIJ)},
	author = {Bischl, Bernd and Kerschke, Pascal and Kotthoff, Lars and Lindauer, Marius and Malitsky, Yuri and Fréchette, Alexandre and Hoos, Holger H. and Hutter, Frank and Leyton-Brown, Kevin and Tierney, Kevin and Vanschoren, Joaquin},
	year = {2016},
	pages = {41--58}
}
""",
    academic_reference_bibtex_key="xu-sat12a,sat12,bischl_aslib_2016",
    license="GPLv3",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We get the data from ASlib and merge them into one file.

- Algorithm selection can be solved with many methods (pairwise classification, hierarchical regression, multi-label, ...). We follow the concept from Pulatov et al. (https://proceedings.mlr.press/v188/pulatov22a.html) and learn one single, unified regression model that goes from (instance_features, algorithm_features) -> runtime.
- The description says 'If features are "?", the instance was solved during feature computation.' but from the status file we can see that no task was presovled. So it is unclear what this is referring to. We believe that in all cases we have nan values for features, it is a case of the feature running into a memout, timeout, or crash as stated by the description.
- This regression task is special as it contains censored regression values (capped to some max value) and the model needs to learn this.
- We log scale the target.
- We drop duplicated columns and remove features that leak the target or are not relevant for the predictive task.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="runtime",
    problem_type="regression",
    objective_metric_name="rmse", # better to score with RMSE with cut-off, or like PAR10?
    # For grouped data
    group_on="instance_id",
    group_labels="per_sample",
)

## Preprocessing

In [ ]:
import arff
import pandas as pd
import uuid
import numpy as np


def load_arff(path) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        data = arff.load(f)
    df = pd.DataFrame(data["data"], columns=[a[0] for a in data["attributes"]])
    return df


df_features = load_arff(dataset_mold.path / "feature_values.arff")
print("feature_values shape:", df_features.shape)
df_algo_features = load_arff(dataset_mold.path / "algorithm_feature_values.arff")
print("algorithm_feature_values shape:", df_algo_features.shape)
df_algo_runs = load_arff(dataset_mold.path / "algorithm_runs.arff")
print("algorithm_runs shape:", df_algo_runs.shape)

# Drop algorithms that for which we do not have features
df_algo_runs = df_algo_runs[df_algo_runs["algorithm"].isin(df_algo_features["algorithm"].unique())]

# Merge
df = df_algo_runs.merge(df_features, on="instance_id", how="left").merge(
    df_algo_features, on="algorithm", how="left"
)
print("Merged data shape:", df.shape)

# Sanitize ID
# Create mapping: molecule -> random string id
mapping = {val: uuid.uuid4().hex[:12] for val in df["instance_id"].unique()}
df["instance_id"] = df["instance_id"].map(mapping)
df["instance_id"] = df["instance_id"].astype("category")

# Drop features
df = df.drop(columns=[
    # Data has no repetitions
    "repetition_x", "repetition_y", "repetition",
    # We want to only rely on algorithm meta features not names.
    "algorithm",
    # Leaks target
    "runstatus",
    # Duplicated columns
    "clustering_min", "edge_ta","edge_to","edge_tl","edge_da","edge_do","edge_dl","edge_as","edge_aa","edge_ao","edge_al","edge_ot","edge_oa","edge_ol","edge_lt","edge_la","edge_ll","degree_min","path_min","VCG_VAR_mean","edge_at",
])

df["runtime"] = np.log(df["runtime"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For Grouped Non-IID data
split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    # For time data
    time_horizon=None,
    time_horizon_unit=None,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()